In [1]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)

rows = 20000
data = []

start_time = pd.Timestamp("2026-01-01")

active_users_prev = 100  # for smooth transitions

for i in range(rows):
    timestamp = start_time + pd.Timedelta(seconds=5*i)
    hour = timestamp.hour

    # Base pattern (time dependent)
    if 8 <= hour <= 10 or 18 <= hour <= 22:
        base_users = np.random.randint(250, 400)
    elif 0 <= hour <= 5:
        base_users = np.random.randint(20, 80)
    else:
        base_users = np.random.randint(80, 200)

    # Smooth transition (time-series realism)
    active_users = 0.7 * active_users_prev + 0.3 * base_users
    active_users += np.random.normal(0, 10)
    active_users_prev = active_users

    # Hidden congestion (NOT directly tied linearly)
    true_congestion = (
        0.6 * (active_users / 400)
        + 0.25 * np.sin(i / 150)
        + 0.15 * np.random.rand()
    )
    true_congestion = np.clip(true_congestion, 0, 1)
    
    # Metrics influenced but not directly predictable
    latency = 30 + 100 * np.tanh(true_congestion * 2) + np.random.normal(0, 10)
    
    packet_loss = max(0, np.random.normal(1 + true_congestion * 3, 0.8))
    
    throughput = max(1, np.random.normal(80 - true_congestion * 50, 10))

    # Signal Strength
    signal_strength = -50 + np.random.normal(0, 4)

    # Random anomalies
    if np.random.rand() < 0.03:
        latency *= np.random.uniform(1.5, 2.5)
        packet_loss *= np.random.uniform(2, 4)
        throughput *= np.random.uniform(0.3, 0.7)

    # Classification label
    if true_congestion < 0.30:
        label = 0
    elif true_congestion < 0.55:
        label = 1
    else:
        label = 2

    data.append([
        timestamp, hour, active_users, latency,
        packet_loss, throughput, signal_strength, label
    ])

df = pd.DataFrame(data, columns=[
    "timestamp", "hour", "active_users", "latency",
    "packet_loss", "throughput", "signal_strength", "congestion_level"
])

# Ensure we save in ml-service directory
save_path = "network_congestion.csv"
if not os.getcwd().endswith("ml-service") and os.path.exists("netpulseai/ml-service"):
    save_path = os.path.join("netpulseai", "ml-service", "network_congestion.csv")

df.to_csv(save_path, index=False)
print(f"Dataset saved as {save_path}")
print(df.head())

Dataset saved as network_congestion.csv
            timestamp  hour  active_users    latency  packet_loss  throughput  \
0 2026-01-01 00:00:00     0     81.897655  79.746673     0.761155   89.426578   
1 2026-01-01 00:00:05     0     65.740078  64.404862     2.053511   65.041440   
2 2026-01-01 00:00:10     0     33.926996  48.218296     1.361334   69.259056   
3 2026-01-01 00:00:15     0     40.202375  54.302033     2.227257   75.088592   
4 2026-01-01 00:00:20     0     47.243945  54.147506     0.356083   70.732656   

   signal_strength  congestion_level  
0       -50.378484                 0  
1       -51.862919                 0  
2       -49.430142                 0  
3       -52.542239                 0  
4       -51.786966                 0  
